This notebook evaluates the global treatment+dose multiclass MLP trained in `6.train_multiclass_mlp_treatment_dose.ipynb`.

It re-derives the held-out test-set predictions (same split, same deterministically-refit `StandardScaler`, saved model weights) and saves: per-cell predictions, overall and per-class accuracy/precision/recall/F1, and top-k accuracy (since a wrong top-1 guess that still lands on the correct mechanism-of-action neighbor, e.g. confusing two MEK inhibitors, is a very different kind of error than a wrong guess altogether). Plotting (including the confusion matrix) is done separately in `8.plot_multiclass_mlp_treatment_dose.ipynb` (R).

In [1]:
import pathlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from notebook_init_utils import init_notebook
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    top_k_accuracy_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

root_dir, in_notebook = init_notebook()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

In [2]:
profile_file_path = pathlib.Path(
    f"{root_dir}/data/profiles_3D/all_patients/1.feature_selected_profiles/sc_norm_fs_profiles.parquet"
).resolve(strict=True)

split_idx_file_path = pathlib.Path(
    f"{root_dir}/6.treatment_predictive_model/data_splits/sc_norm_fs_profiles_split_idx.parquet"
).resolve(strict=True)

model_output_dir = pathlib.Path(
    f"{root_dir}/6.treatment_predictive_model/trained_models"
).resolve(strict=True)
model_results_dir = pathlib.Path(
    f"{root_dir}/6.treatment_predictive_model/model_results"
).resolve()
model_results_dir.mkdir(parents=True, exist_ok=True)

## Rebuild the same data/label setup used in training
This mirrors `6.train_multiclass_mlp_treatment_dose.ipynb` exactly, so refitting the `StandardScaler` and `LabelEncoder` on `train_df` reproduces what the saved model weights were trained under.

In [3]:
TREATMENT_COL = "Metadata_Experiment_Treatment"
DOSE_COL = "Metadata_Experiment_Dose"
LABEL_COL = "Metadata_treatment_dose_label"

indices_df = pd.read_parquet(split_idx_file_path)
df = pd.read_parquet(profile_file_path)

feature_cols = [col for col in df.columns if not col.startswith("Metadata_")]
df[LABEL_COL] = df[TREATMENT_COL].astype(str) + "_" + df[DOSE_COL].astype(str)

train_idx = indices_df["train_idx"].dropna().astype(int).tolist()
test_idx = indices_df["test_idx"].dropna().astype(int).tolist()
train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]

label_encoder = LabelEncoder()
label_encoder.fit(train_df[LABEL_COL])
num_classes = len(label_encoder.classes_)

scaler = StandardScaler()
scaler.fit(train_df[feature_cols].values)
X_test = scaler.transform(test_df[feature_cols].values)
y_test = label_encoder.transform(test_df[LABEL_COL])

print(f"Test: {len(test_df)} rows, {num_classes} classes")

Test: 8857 rows, 26 classes


## Model architecture
Must match `6.train_multiclass_mlp_treatment_dose.ipynb` exactly so the saved `state_dict` loads correctly.

In [4]:
HIDDEN_DIMS = (256, 128)


class MulticlassMLP(nn.Module):
    """A feed-forward multiclass classifier over single-cell morphology features."""

    def __init__(
        self, input_dim: int, num_classes: int, hidden_dims: tuple = HIDDEN_DIMS
    ):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers += [
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
            ]
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = MulticlassMLP(input_dim=len(feature_cols), num_classes=num_classes).to(device)
model.load_state_dict(
    torch.load(
        model_output_dir / "multiclass_mlp_treatment_dose.pt", map_location=device
    )
)
model.eval()

MulticlassMLP(
  (network): Sequential(
    (0): Linear(in_features=386, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=128, out_features=26, bias=True)
  )
)

## Score the test set

In [5]:
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test, dtype=torch.float32),
        torch.tensor(y_test, dtype=torch.long),
    ),
    batch_size=128,
    shuffle=False,
)

all_logits, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        all_logits.append(model(X_batch.to(device)).cpu())
        all_labels.append(y_batch)
logits = torch.cat(all_logits).numpy()
labels = torch.cat(all_labels).numpy()
preds = logits.argmax(axis=1)
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

test_predictions_df = pd.DataFrame(
    {
        "Metadata_true_label": label_encoder.inverse_transform(labels),
        "Metadata_predicted_label": label_encoder.inverse_transform(preds),
    }
)
test_predictions_df.to_parquet(
    model_results_dir / "multiclass_mlp_treatment_dose_test_predictions.parquet",
    index=False,
)
test_predictions_df.head()

,Metadata_true_label,Metadata_predicted_label
0,Trabectedin_1.0,Trabectedin_1.0
1,Digoxin_1.0,Digoxin_1.0
2,Selumetinib_1.0,Selumetinib_1.0
3,Imatinib_1.0,Imatinib_1.0
4,Mirdametinib_1.0,Mirdametinib_10.0


## Overall metrics and top-k accuracy

In [6]:
top_k_values = [1, 2, 3, 5]
top_k_rows = [
    {
        "k": k,
        "top_k_accuracy": top_k_accuracy_score(
            labels, probs, k=k, labels=np.arange(num_classes)
        ),
    }
    for k in top_k_values
]
top_k_df = pd.DataFrame(top_k_rows)
top_k_df.to_parquet(
    model_results_dir / "multiclass_mlp_treatment_dose_top_k_accuracy.parquet",
    index=False,
)

print(f"Accuracy: {accuracy_score(labels, preds):.4f}")
print(f"Macro F1: {f1_score(labels, preds, average='macro'):.4f}")
print(f"Weighted F1: {f1_score(labels, preds, average='weighted'):.4f}")
top_k_df

Accuracy: 0.2485
Macro F1: 0.2562
Weighted F1: 0.2277


,k,top_k_accuracy
0,1,0.248504
1,2,0.373829
2,3,0.466524
3,5,0.608784


## Per-class precision/recall/F1

In [7]:
report_dict = classification_report(
    labels,
    preds,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
report_df = (
    pd.DataFrame(report_dict)
    .T.reset_index(names="Metadata_class")
    .query("Metadata_class not in ['accuracy']")
)
report_df.to_parquet(
    model_results_dir / "multiclass_mlp_treatment_dose_classification_report.parquet",
    index=False,
)
report_df.sort_values("f1-score", ascending=False)

,Metadata_class,precision,recall,f1-score,support
0,ARV-825_1.0,0.379195,0.617486,0.469854,183.0
21,Staurosporine_10.0,0.412671,0.476285,0.442202,506.0
6,Digoxin_1.0,0.375335,0.403458,0.388889,347.0
22,Trabectedin_1.0,0.273256,0.610390,0.377510,154.0
20,Selumetinib_10.0,0.242038,0.368932,0.292308,309.0
2,Binimetinib_10.0,0.315113,0.253886,0.281205,386.0
16,Panobinostat_1.0,0.191748,0.478788,0.273830,165.0
15,Onalespib_1.0,0.257534,0.277286,0.267045,339.0
9,Imatinib_1.0,0.197339,0.372385,0.257971,239.0
27,macro avg,0.248169,0.306265,0.256203,8857.0
